<a href="https://colab.research.google.com/github/Sravani-939/genai-training-tasks/blob/main/audio_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install -q SpeechRecognition pydub
!apt-get -qq install -y ffmpeg


In [32]:


from google.colab.output import eval_js
from base64 import b64decode
import speech_recognition as sr
from pydub import AudioSegment

def record_audio(seconds=5, filename="recorded.wav"):
    js = f"""
    async function recordAudio(seconds) {{
      const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      await new Promise(resolve => setTimeout(resolve, seconds * 1000));

      recorder.stop();

      const blob = await new Promise(resolve => {{
        recorder.onstop = () => resolve(new Blob(chunks, {{type: 'audio/webm'}}));
      }});

      const arrayBuffer = await blob.arrayBuffer();
      const base64Audio = btoa(String.fromCharCode(...new Uint8Array(arrayBuffer)));

      stream.getTracks().forEach(track => track.stop());
      return base64Audio;
    }}
    recordAudio({seconds});
    """

    audio_base64 = eval_js(js)
    audio_data = b64decode(audio_base64)

    with open("recorded.webm", "wb") as f:
        f.write(audio_data)

    audio = AudioSegment.from_file("recorded.webm", format="webm")
    audio.export(filename, format="wav")

    return filename

audio_file = record_audio(seconds=5)
print("Recording complete:", audio_file)

recognizer = sr.Recognizer()

with sr.AudioFile(audio_file) as source:
    audio = recognizer.record(source)

try:
    text = recognizer.recognize_google(audio)
    print("You said:", text)
except sr.UnknownValueError:
    print("Sorry, could not understand the audio.")
except sr.RequestError as e:
    print("Speech recognition error:", e)

Recording complete: recorded.wav
You said: hello
